# Práctica 4: Umbralización

En esta práctica vamos a programar algunos de los algoritmos vistos en clase sobre umbralización. El objetivo es seguir afianzando los conocimientos de programación y tener una visión más clara sobre qué hace cada uno de los algoritmos vistos en clase

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### Ejercicio 1: mínimo entre máximos

El primer programa que vamos a implementar es la umbralización de mínimo entre máximos. Para ello, crea una función que reciba como parámetro de entrada una imagen. El programa obtendrá el histograma de la imagen. A partir del histograma es muy sencillo obtener la posición que ocupa el máximo global del histograma (recuerda la función argmax). El siguiente paso es encontrar el segundo pico del histograma. Recuerda que no es tan sencillo como buscar el segundo máximo del histograma, sino que tenemos que recorrer cada una de las intensidades:
- calcular el pico del histograma $q_1=\arg \max_i h(i)$
- para cada intensidad i:
    - Calcular $h(i)\times |i-q_1|$
- tomar como $q_2$  la posición que maximice la expresión anterior

Una vez localizados $q_1,q_2$, busca el mínimo del histograma entre ambos picos ($q_2$ puede quedar a la izquierda o a la derecha de $q_1$). Umbraliza la imagen utilizando dicho umbral. Recuerda que puedes hacer algo como
 $$umbralizada = 255*(imagen<=q).astype(np.uint8)$$
 

In [2]:
def min_entre_max(imagen):
    hist = cv2.calcHist([imagen], [0], None, [256],[0,256]).ravel()
    q1 = np.argmax(hist) 
    restas = np.zeros(256)
    i = np.arange(0,256)
    q2 = np.argmax(hist * np.abs(i-q1))
    a, b = min(q1, q2), max(q1, q2)
    q = a+ np.argmin(hist[a:b+1])
    umbralizada = 255 * (imagen<= q).astype(np.uint8)
    return umbralizada

Una vez programada llama a la función con la imagen 01.bmp. Si devuelves el primer mínimo que encuentres, deberías tener como umbral $q=56$ ¿Obtienes un buen resultado? ¿Por qué? ¿Qué pegas encuentras?

In [3]:
imagen = cv2.imread('images/01.bmp', 0)
print("ha leido imagen")
umbralizada = min_entre_max(imagen)
print("Umbralizada")
cv2.imshow('Umbralizacion minimo entre maximos', umbralizada)
cv2.waitKey(0)
cv2.destroyAllWindows()

ha leido imagen
Umbralizada


### Ejercicio 2: Otsu

Realizar una función en python que implemente el algoritmo de Otsu. Para ello, vamos a crear un array vacío de 256 posiciones. En la posición q-ésima del vector almacenaremos la varianza entre clases $\sigma_b(q)$ tomando como umbral el valor de intensidad q (probaremos $q=0,1,\dots,254$, para que ninguna de las dos clases quede vacía). Por último obtendremos como umbral aquel asociado a la máxima varianza entre clases. Recuerda que para calcular la varianza entre clases dado q debemos calcular:
- $p_0$: probabilidad de la clase 0
- $p_1$: probabilidad de la clase 1
- $\mu_0$: media de la clase 0
- $\mu_1$: media de la clase 1
- varianza entre clases: $\sigma_b^2(q)=p_0p_1(\mu_0- \mu_1)^2$

Estos cálculos se pueden hacer de dos maneras distintas. Una opción sería utilizando funciones de numpy como np.mean, etc. Otra opción sería utilizar el histograma de la imagen y calcular $p_0,p_1,\mu_0,\mu_1$ utilizando únicamente el histograma. Puedes elegir aquella que te resulte más sencilla.

In [ ]:
def otsu(imagen):
    #hist normalizado
    h = cv2.calcHist([imagen], [0], None, [256],[0,256]).ravel()
    hp = h/h.sum()
    i = np.arange(0,256)

    varianzas = np.zeros(256)
    for q in range(varianzas.size):
        p0 = hp[:q+1].sum()
        p1 = 1-p0
        if p1==0 or p0==0:
            continue
        m0 = (1/hp)
    umbralizada = 255 * (imagen<= q).astype(np.uint8)
    return umbralizada

Una vez programada, prueba la umbralización con la imagen 01.bmp. Comprueba que obtienes el umbral 143.

Compara las imagenes obtenidas con el Ejercicio 1 y el Ejercicio 2. ¿Por qué crees que el resultado del Ejercicio 1 no tiene tanta calidad?

Prueba ahora a suavizar la imagen con un filtro Gaussiano (o un filtro de la media 3x3 o 5x5) y luego umbraliza con la función Otsu. ¿Obtienes alguna mejora?


In [ ]:
imagen = cv2.imread('images/01.bmp', 0)
umbralizada = otsu(imagen)
cv2.imshow('Umbralizacion Otsu', umbralizada)
cv2.waitKey(0)
cv2.destroyAllWindows()

### Ejercicio 3: Umbralización de mínimo error

Este algoritmo es algo parecido al anterior. Vamos a ir probando cada una de las intensidades posibles como umbral y, para cada una, calcularemos una distribución de probabilidad gaussiana según la media y la desviación de los píxeles de la clase 0 y clase 1 en función del umbral. Para ello, recorre el conjunto de intensidades y calcula:
- $p_0$: probabilidad de la clase0
- $p_1$: probabilidad de la clase1
- $\mu_0$: media de la clase 0
- $\mu_1$: media de la clase 1
- $\sigma_0^2$: varianza de la clase 0
- $\sigma_1^2$: varianza de la clase 1


Una vez tenemos los parámetros de cada una de las clases, vamos a calcular el error que cometemos al asignar cada una de las intensidades a la clase 0 y a la clase 1 en función del umbral. Esto se calcula como

$$e(q)=\sum_{i=0}^{q}h_p(i)e_0(i)+\sum_{i=q+1}^{255}h_p(i)e_1(i)$$

donde $h_p$ es el histograma normalizado y

$$e_j(i)=\frac{(i-\mu_j)^2}{\sigma_j^2}+2(\log(\sigma_j)-\log(p_j))$$

Nota: en el cálculo de las varianzas, podemos sumarle a la variable un pequeño valor (1e-10) para evitar el logaritmo de 0.

Una vez calculado el error por cada posible umbral, tomaremos aquel que minimice el error cometido.

In [ ]:
def minimo_error(imagen):
    
    umbralizada = 255 * (imagen<= q).astype(np.uint8)
    return umbralizada

Una vez programado, comprueba su funcionamiento con las imágenes de antes y revisa si obtienes umbral 78. Al igual que aparece en las explicaciones de clase, podríamos pintar encima del histograma las dos distribuciones gaussianas que tratan de ajustarse al histograma. Para ello, una vez conocido el umbral q, las distribuciones gaussianas de la clase 0 y de la clase 1 vendrían dadas, para cada posible intensidad x, por

$$\frac{1}{\sqrt{2\pi\sigma_0^2}}*\exp\left(-\frac{(x-\mu_0)^2}{2\sigma_0^2}\right)$$
$$\frac{1}{\sqrt{2\pi\sigma_1^2}}*\exp\left(-\frac{(x-\mu_1)^2}{2\sigma_1^2}\right)$$

In [ ]:
imagen = cv2.imread('images/01.bmp', 0)
umbralizada = minimo_error(imagen)
cv2.imshow('Umbralizacion minimo error', umbralizada)
cv2.waitKey(0)
cv2.destroyAllWindows()

### Ejercicio 4: Umbralización variable

Uno de los problemas del algoritmo de Otsu aparece cuando la iluminación de la imagen no es constante. Lee la imagen texto.tif. Visualiza y aplica el algoritmo del ejercicio 2 (otsu). Comprueba que la segmentación no es la adecuada. Vamos a solventar este problema mediante una umbralización variable, es decir, una combinación de técnicas globales y locales. De hecho, vamos a ver que se parece mucho a un filtro no lineal sobre la imagen. El algoritmo es el siguiente:
- para cada píxel de la imagen
    - tomar una ventana 3x3 de píxeles centrada en el
    - calcular la media aritmética $\mu_{x,y}$ y la desviación típica $\sigma_{x,y}$
    - Realizar la siguiente segmentación 
    $$g(x,y)=\begin{cases} 
        1 & \mbox{ si } imagen(x,y)<\sigma_{x,y}+0.5\mu_{x,y} \\
        0 & \mbox{ en otro caso}
    \end{cases}$$

La función debe devolver $g$ como una imagen de tipo np.uint8 (con valores 0 y 1).

In [ ]:
imagen = cv2.imread('images/texto.tif', 0)
cv2.imshow('Imagen texto', imagen)
cv2.waitKey(0)
cv2.destroyAllWindows()

texto_umbralizada = otsu(imagen)
cv2.imshow('Otsu sobre texto', texto_umbralizada)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
def variable(imagen):
    
    return g

Comprueba ahora que la umbralización variable funciona bien en este ejemplo concreto. Evidentemente, la fórmula elegida es ad-hoc para la imagen concreta, pero en función de la iluminación podríamos elegir otras condiciones del estilo 
$$imagen(x,y)<a \sigma_{x,y} + b \mu_{x,y}$$
siendo $a$ y $b$ dos números cualesquiera.

In [ ]:
imagen = cv2.imread('images/texto.tif', 0)
texto_umbralizada_2 = variable(imagen)*255
cv2.imshow('Umbralizacion Variable', texto_umbralizada_2)
cv2.waitKey(0)
cv2.destroyAllWindows()